In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [2]:
# Đọc dữ liệu
df = pd.read_csv('./Products.csv')
df.head(3)

,ProductKey,Product Name,Brand,Color,Unit Cost USD,Unit Price USD,SubcategoryKey,Subcategory,CategoryKey,Category
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,$6.62,$12.99,101,MP4&MP3,1,Audio
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,$6.62,$12.99,101,MP4&MP3,1,Audio
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,$7.40,$14.52,101,MP4&MP3,1,Audio


In [3]:
main_feature4m_df = df[['ProductKey','Product Name', 'Brand', 'Color','Subcategory', 'Category']]

In [4]:
df = df.dropna()

In [5]:
import re

main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace('MP4&MP3', 'MP4 and MP3')
main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace(' & ', ' and ')
main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace('Printers, Scanners and Fax', 'Printers Scanners and Fax')

In [6]:
main_feature4m_df['product_details'] = (
    main_feature4m_df['Product Name'] + ' ' + 
    main_feature4m_df['Brand'] + ' ' + 
    main_feature4m_df['Color'] + ' ' + 
    main_feature4m_df['Subcategory'] + ' ' + 
    main_feature4m_df['Category'] 
)

In [7]:
# Tính toán TF-IDF
tfidf_vec = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2, ngram_range=(1, 1))
tfidf_matrix = tfidf_vec.fit_transform(main_feature4m_df['product_details'])

In [8]:
# Tính toán K-NN
knn = NearestNeighbors(n_neighbors=5, metric='cosine')  # Sử dụng cosine distance
knn.fit(tfidf_matrix)

NearestNeighbors(metric='cosine')

In [9]:
# Hàm gợi ý tài liệu
def content_based_knn_recommendations(paper_index, data, knn_model):
    distances, indices = knn_model.kneighbors(tfidf_matrix[paper_index], n_neighbors=10)  
    return data.iloc[indices.flatten()[1:]], distances.flatten()[1:]

# Ví dụ gọi hàm với chỉ số bài báo
input_product_index = int(input("index of the pd : "))
recommendations_result, distances = content_based_knn_recommendations(input_product_index, main_feature4m_df, knn)

# Hiển thị kết quả
print("Recommendations:")
print(recommendations_result[['product_details']])

Recommendations:
                                      product_details
0   Contoso 512MB MP3 Player E51 Silver Contoso Si...
6   Contoso 2G MP3 Player E200 Blue Contoso Blue M...
11  Contoso 4GB Flash MP3 Player E401 Blue Contoso...
2   Contoso 1G MP3 Player E100 White Contoso White...
20  Contoso 8GB MP3 Player new model M820 Blue Con...
39  Contoso 8GB Clock & Radio MP3 Player X850 Blue...
5   Contoso 2G MP3 Player E200 Black Contoso Black...
3   Contoso 2G MP3 Player E200 Silver Contoso Silv...
23  Contoso 16GB Mp5 Player M1600 Blue Contoso Blu...
